# Estimator Sheet 
This notebook builds the single combined dataset used for quantile regression estimation. It walks a folder of forecast-card CSVs (`project-*.csv`, `poi-*.csv`, `observations-*.csv`), finds every project, combines each project's files into one merged table, adds K/D factors, county-level socioeconomic data, peak-hour volume, and post-construction capacity, then writes everything out to a single `combined_output.csv`. That combined file is the input for the model-fitting notebook.

**How to use this notebook:**
1. Run the "find project folders" cell to confirm every project was located. Set `DATA_ROOT` to the folder containing all your project CSVs.
2. Run the "build the combined dataset" cell. For each project this loads its project, poi, and observations CSVs, merges them together, and labels each observation as `before` or `after` the project's actual open date. All projects are then stacked into one `master_df`.
3. Run the "years from open" cell to add a `years_from_open` column to `master_df`, the difference between each observation's year and the project's actual open year.
4. Run the "K/D factors" cell to add `k_factor` and `d_factor` to `master_df`, joined from `K_D_Factors_by_State_and_Year.csv` on state, area type, functional class, and observation year.
5. Run the "add county-level socioeconomic data" cell to merge in population, employment, pay, and fuel-price columns from `us_county_socioeconomic_1995_2025.csv` (matched on state, county, and year). These columns land right after `d_factor` and are also what the capacity equation's population/income growth terms pull from.
6. Run the "peak-hour volume" cell to add `peak_hour_volume`. It runs after the socioeconomic merge so that, in `combined_output.csv`, the socioeconomic columns end up sitting between `d_factor` and `peak_hour_volume`.
7. Run the "post-construction capacity" cells to add `engineering_capacity_ceng`, `capacity_multiplier`, and `peak_hour_capacity` to `master_df`, via the **Post-Construction Capacity Expansion Equation**.
8. Run the "spot-check" cell to visually confirm the merges and factor calculations worked as expected.
9. Run the "write out" cell to add a blank `comment` column and save `master_df` to `combined_output.csv`.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Find your project folders

Point `DATA_ROOT` at the folder holding your project data. This walks the entire directory tree underneath it and collects every sub-folder that contains a `project-*.csv` file (each matching folder is treated as one project). The result is a sorted list of folder paths, printed below so you can confirm the right projects were found.

**Update this path to point at your own data folder before continuing.**

In [2]:
def find_project_folders(root_dir):
    """Return every sub-folder of root_dir that contains a project-*.csv file."""
    folders = []
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        if any(re.match(r"project-.*\.csv$", f, re.IGNORECASE) for f in filenames):
            folders.append(dirpath)
    return sorted(folders)

DATA_ROOT = r"C:\Users\User\Documents\Code\forecastcards_ba_data\data"
project_folders = find_project_folders(DATA_ROOT)
project_folders

['C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237506-2',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237565-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237576-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237592-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237592_2',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237650-3',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237965-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238394-3',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238395-4',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238395-5',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238429-4',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238429-5',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba

## 2. Build the combined dataset

This section does the actual finding, loading, and merging for every project, then stacks the results into one master table used for estimation.

**Loading a project's files** — for a given project folder, finds and reads:
- `project-*.csv` → project-level metadata (one row per project)
- `poi-*.csv` → "points of interest" tied to that project (e.g. intersections/segments)
- `observations-*.csv` → observed traffic/volume records

**Merging a project's files** — performs two sequential merges:
- **project → poi:** tags each POI row with its `project_id`, then merges project metadata onto every POI row (keeping all POIs, broadcasting the single project row across them).
- **before/after labeling:** converts `observation_date` to a real datetime and compares it to the project's `date_open_actual` to create `before_or_after`.
- **(project+poi) → observations:** merges the project/poi table onto the observations, matching on `poi_id` **and** `before_or_after`, keeping every observation row, and renames `start_time`/`end_time` to the `_observation` versions for consistent naming.

**Dates:** each card's open dates are parsed once, here, per project (cards differ: FL writes `1/1/2003`, KY writes `2018-01-01`),
and stored as real dates on every row. Section 3 used to re-parse all projects' dates together as text, and pandas applied the first
row's format (`1/1/2003`) to everyone, silently blanking every `2018-01-01`-style date (most KY planned open dates).

In [3]:
def read_csv_any_encoding(path, **kwargs):
    """Try UTF-8 first; fall back to Windows-1252 (Excel's default on Windows)."""
    for enc in ("utf-8", "utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            if enc == "cp1252":
                print(f"Note: read {os.path.basename(path)} as cp1252")
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"Could not decode {path}")
    # non-breaking spaces in headers would break lookups like df["project_id"]
    df.columns = df.columns.str.replace("\xa0", " ").str.strip()
    # drop blank header-less columns left by trailing commas (e.g. "Unnamed: 8")
    unnamed = [c for c in df.columns if c.startswith("Unnamed:") and df[c].isna().all()]
    df = df.drop(columns=unnamed)
    return df


# Some projects use different names for the same field. Map them onto one name
# so they land in a single column in combined_output.csv.
COLUMN_ALIASES = {
    "milepoint": "segment_length_miles",
    "speed_limit_mph": "speed_limit",
}

def apply_column_aliases(df):
    for old, new in COLUMN_ALIASES.items():
        if old not in df.columns:
            continue
        if new in df.columns:
            df[new] = df[new].fillna(df[old])
            df = df.drop(columns=old)
        else:
            df = df.rename(columns={old: new})
    return df


def load_project_files(folder):
    """Load the project / poi / observations csvs for one folder."""
    project_path = glob.glob(os.path.join(folder, "project-*.csv"))[0]
    poi_path = glob.glob(os.path.join(folder, "poi-*.csv"))[0]
    obs_path = glob.glob(os.path.join(folder, "observations-*.csv"))[0]

    project_df = read_csv_any_encoding(project_path).dropna(how="all")
    poi_df = read_csv_any_encoding(poi_path, dtype={"poi_id": str}).dropna(how="all")
    obs_df = read_csv_any_encoding(obs_path, dtype={"poi_id": str}).dropna(how="all")

    project_df = apply_column_aliases(project_df)
    poi_df = apply_column_aliases(poi_df)
    obs_df = apply_column_aliases(obs_df)

    return project_df, poi_df, obs_df

In [4]:
def process_project_folder(folder):
    project_df, poi_df, obs_df = load_project_files(folder)

    project_df = project_df.copy()

    comment_cols = [c for c in project_df.columns if c.strip().lower() in ("comment", "comments")]
    if comment_cols:
        combined_comment = project_df[comment_cols[0]]
        for extra_col in comment_cols[1:]:
            combined_comment = combined_comment.fillna(project_df[extra_col])
        project_df = project_df.drop(columns=comment_cols)
        project_df["comment"] = combined_comment
    else:
        project_df["comment"] = pd.NA

    project_id = project_df["project_id"].iloc[0]
    date_open_actual = pd.to_datetime(project_df["date_open_actual"].iloc[0], errors="coerce", format="mixed")
    date_open_planned = pd.to_datetime(project_df["date_open_planned"].iloc[0], errors="coerce", format="mixed")

    # use the actual open date when available; otherwise fall back to the planned open date
    date_open_effective = date_open_actual if pd.notna(date_open_actual) else date_open_planned

    if pd.isna(date_open_effective):
        print(f"WARNING: no usable open date for project {project_id} in {folder}")

    #match POIs ignoring leading zeros
    def poi_key(s):
        return s.astype("string").str.strip().str.lstrip("0")

    poi_df = poi_df.copy()

    if "before_or_after" in poi_df.columns:
        poi_df["before_or_after"] = (
            poi_df["before_or_after"].astype("string").str.strip().str.lower()
        )
    poi_df["project_id"] = project_id
    poi_df["_poi_key"] = poi_key(poi_df["poi_id"])                       

    # STEP 1: project -> poi
    project_poi = pd.merge(project_df, poi_df, on="project_id", how="right")

    # derive a sensible before/after label for each observation
    obs_df = obs_df.copy()
    # format="mixed": parse each date on its own, so one file mixing 1/1/2020 and 2020-01-01 doesn't lose rows
    obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"], errors="coerce", format="mixed")
    obs_df = obs_df.dropna(subset=["observation_date"])
    obs_df["before_or_after"] = pd.array(
        np.where(obs_df["observation_date"] < date_open_effective, "before", "after"),
        dtype="string",
    )
    obs_df["before_or_after_binary"] = (obs_df["before_or_after"] == "after").astype(int)
    obs_df["_poi_key"] = poi_key(obs_df["poi_id"])                       

    # STEP 2: (project + poi) -> observation
    # keep every observation row; observations with no matching POI get blank POI columns
    merged = pd.merge(
        project_poi, obs_df,
        on=["_poi_key", "before_or_after"],
        how="right",
        suffixes=("", "_obs"),
        indicator=True,
    )
    merged["poi_id"] = merged["poi_id_obs"]

    # unmatched rows came through with blank project columns too -- refill those,
    # since the project info is the same for every row in this folder
    for col in project_df.columns:
        if col in merged.columns:
            merged[col] = project_df[col].iloc[0]

    unmatched = merged[merged["_merge"] == "right_only"]
    if len(unmatched) > 0:
        print(f"Project {project_id}: kept {len(unmatched)} observation rows with no matching POI "
              f"(POI columns left blank): {sorted(unmatched['poi_id'].dropna().unique())}")

    merged = merged.drop(columns=["_poi_key", "poi_id_obs", "_merge"])

    merged = merged.rename(columns={"start_time": "start_time_observation",
                                     "end_time": "end_time_observation"})

    # store the open dates already parsed above (as real dates), so they never have to be re-parsed from text later
    merged["date_open_actual"] = date_open_actual
    merged["date_open_planned"] = date_open_planned
    merged["date_open_effective"] = date_open_effective

    return merged

In [5]:
project_tables = [process_project_folder(folder) for folder in project_folders]

master_df = pd.concat(project_tables, ignore_index=True, sort=False)
# project_id as text everywhere (some cards have numeric ids like 20200057), since it is used as a grouping key below
master_df["project_id"] = master_df["project_id"].astype(str).str.strip()

print(f"{len(master_df)} total observation rows across {len(project_tables)} project(s)")
master_df.head(10)

Note: read poi-238394-3.csv as cp1252
Project 238429-4: kept 14 observation rows with no matching POI (POI columns left blank): ['111012']
Project 238753-1A: kept 28 observation rows with no matching POI (POI columns left blank): ['360024', '360030']
Project 240805-2: kept 19 observation rows with no matching POI (POI columns left blank): ['791011']
Project 6-313.50: kept 9 observation rows with no matching POI (POI columns left blank): ['059G10']
Project 6-356.00: kept 9 observation rows with no matching POI (POI columns left blank): ['094A04']
Project 7-113.01: kept 46 observation rows with no matching POI (POI columns left blank): ['034E49', '034E54', '034E58']
Project 7-8405.00: kept 10 observation rows with no matching POI (POI columns left blank): ['009A80']
Project 20200057: kept 9 observation rows with no matching POI (POI columns left blank): ['02489002', '02489004', '02489005', '02489006', '02489007', '02489008', '02489009', '02489010', '02489011']
6681 total observation rows

,project_id,state,county,project_type,description,date_open_planned,date_horizon,construction_start_date,date_open_actual,area_type,project_length_miles,comment,poi_id,before_or_after,facility_name,segment_length_miles,speed_limit,lane_width,signals_per_mile,stop_signs_per_mile,through_lanes,left_turn_lanes,right_turn_lanes,express_hov_hot_lanes,shoulder_width,functional_class,obs_id,observation_date,start_time_observation,end_time_observation,observation_variable,observation_value,observation_unit,before_or_after_binary,date_open_effective
0,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_1,1997-01-01,0:00:00,24:00:00,volume,29800,AADT,0,2007-01-01
1,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_2,1998-01-01,0:00:00,24:00:00,volume,30500,AADT,0,2007-01-01
2,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_3,1999-01-01,0:00:00,24:00:00,volume,30100,AADT,0,2007-01-01
3,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_4,2000-01-01,0:00:00,24:00:00,volume,30500,AADT,0,2007-01-01
4,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_5,2001-01-01,0:00:00,24:00:00,volume,30500,AADT,0,2007-01-01
5,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_6,2010-01-01,0:00:00,24:00:00,volume,27000,AADT,1,2007-01-01
6,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_7,2011-01-01,0:00:00,24:00:00,volume,27000,AADT,1,2007-01-01
7,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_8,2012-01-01,0:00:00,24:00:00,volume,22500,AADT,1,2007-01-01
8,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_9,2013-01-01,0:00:00,24:00:00,volume,26000,AADT,1,2007-01-01
9,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",2003-01-01,1/1/2023,unknown,2007-01-01,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_10,2014-01-01,0:00:00,24:00:00,volume,25500,AADT,1,2007-01-01


## 3. Years from open (before/after completion year)

Adds a `years_from_open` column to `master_df`: the difference between the calendar year of each observation and the calendar year the project opened: `date_open_actual` if known, otherwise `date_open_planned` (`date_open_effective`).

In [6]:
# The open dates were already parsed per project in Section 2. format="mixed" parses each value on its own: without it,
# pandas applies the FIRST row's format to the whole column (e.g. 1/1/2003 from an FL card) and silently blanks every
# value written differently (e.g. 2018-01-01 from a KY card).
for _c in ["observation_date", "date_open_actual", "date_open_planned"]:
    _raw = master_df[_c]
    master_df[_c] = pd.to_datetime(_raw, errors="coerce", format="mixed")
    _lost = _raw.notna() & ~_raw.astype(str).str.strip().str.lower().isin(["", "none", "nan", "nat", "unknown"]) & master_df[_c].isna()
    if _lost.any():
        print(f"WARNING: {_lost.sum()} {_c} value(s) could not be read as dates, e.g. {_raw[_lost].unique()[:5].tolist()}")

# use the actual open date when available; otherwise fall back to the planned open date
master_df["date_open_effective"] = master_df["date_open_actual"].fillna(master_df["date_open_planned"])

_dated = master_df.groupby("project_id")["date_open_effective"].apply(lambda s: s.notna().any())
print(f"Projects with an open date: {_dated.sum()} of {len(_dated)} "
      f"(no date: {', '.join(sorted(_dated.index[~_dated])) or 'none'})")

master_df["years_from_open"] = (
    master_df["observation_date"].dt.year - master_df["date_open_effective"].dt.year
)

master_df[["project_id", "poi_id", "observation_date", "date_open_actual", "date_open_planned", "date_open_effective", "years_from_open"]].head(10)

Projects with an open date: 175 of 177 (no date: 20190293, 20200057)


,project_id,poi_id,observation_date,date_open_actual,date_open_planned,date_open_effective,years_from_open
0,237506-2,701007,1997-01-01,2007-01-01,2003-01-01,2007-01-01,-10.0
1,237506-2,701007,1998-01-01,2007-01-01,2003-01-01,2007-01-01,-9.0
2,237506-2,701007,1999-01-01,2007-01-01,2003-01-01,2007-01-01,-8.0
3,237506-2,701007,2000-01-01,2007-01-01,2003-01-01,2007-01-01,-7.0
4,237506-2,701007,2001-01-01,2007-01-01,2003-01-01,2007-01-01,-6.0
5,237506-2,701007,2010-01-01,2007-01-01,2003-01-01,2007-01-01,3.0
6,237506-2,701007,2011-01-01,2007-01-01,2003-01-01,2007-01-01,4.0
7,237506-2,701007,2012-01-01,2007-01-01,2003-01-01,2007-01-01,5.0
8,237506-2,701007,2013-01-01,2007-01-01,2003-01-01,2007-01-01,6.0
9,237506-2,701007,2014-01-01,2007-01-01,2003-01-01,2007-01-01,7.0


## 4. K/D factors

Adds `k_factor` and `d_factor` from `K_D_Factors_by_State_and_Year.csv`, joined on **state + area type + functional class + year**:

- **State**: the row's `state` (two-letter code). States that aren't in the table use its national (`US`) rows.
- **Area type**: `area_type_normalized` mapped onto the table's Urban / Rural (`KD_AREA_MAP`: urban, suburban and exurban → Urban; rural → Rural). Edit the map if a label should go the other way.
- **Functional class**: `functional_class` normalized with `normalize_functional_class` (freeway / principal arterial / minor arterial / major collector / minor collector / local / ramp).
- **Year**: the calendar year of `observation_date`. Years outside the table's 1995–2025 range use the nearest year.

Also adds `kd_year_match` to `master_df` (the table's *Year Match* column: whether that state's value was in effect/measured for that year, or carried back/forward from another year). It's for checking only and is dropped before `combined_output.csv` is written, so the output columns are unchanged.

Rows with an unknown area type or `functional_class == "unknown"` come out blank for these columns; the cell prints a count of blank rows by reason.

Two adjustments on top of the table lookup:

- **Unknown functional class**: the class is first filled from the facility name (`I-75`, `Interstate`, `Expressway` → freeway; names containing `RAMP` → ramp). Rows still unknown get the average K and D of the arterial and collector classes for the same state, area type and year (`KD_FILL_UNKNOWN_CLASS = True`; set it to `False` to leave them blank).
- **Ramps**: `d_factor` = 1.0. Ramp counts are already one-way, so no directional split applies.


In [7]:
# Paths to the factor tables
FACTOR_TABLES_ROOT = r"C:\Users\User\Documents\Code\forecastcards_v2\forecastcards\capacity_factor_tables"

K_D_FACTORS_PATH = os.path.join(FACTOR_TABLES_ROOT, "K_D_Factors_by_State_and_Year.csv")
CAP_CAL_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Cal_Values.csv")
CAP_BASE_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Base_Values.csv")

# The canonical functional-class buckets shared by K_D_Factors_by_State_and_Year.csv and Cap_Base_Values.csv.
CANONICAL_FUNCTIONAL_CLASSES = {
    "freeway", "principal arterial", "minor arterial",
    "major collector", "minor collector", "local", "ramp",
}

def normalize_functional_class(value):
    """Normalize a functional_class label onto the canonical set of classes shared by
    K_D_Factors.csv and Cap_Base_Values.csv (freeway / principal arterial / minor arterial /
    major collector / minor collector / local / ramp). Handles:
      - case/whitespace differences
      - Cap_Base_Values.csv's "Local Road" vs K_D_Factors.csv's "Local"
      - KY's "<Class> - <Qualifier>" suffixes, e.g. "Principal Arterial - Other"
      - "Interstate" as a synonym for "Freeway"
      - a known data-entry typo, "Princicpal Arterial"
    Anything that still doesn't match a canonical class after this (e.g. "unknown") is
    returned as-is, and will simply come out blank in the k_factor/d_factor/Cap_Base lookups.
    """
    if pd.isna(value):
        return None
    label = str(value).strip().lower()

    if "freeways and expressway" in label:
        return "freeway"

    if " - " in label:
        label = label.split(" - ")[0].strip()

    # known data-entry typo ("Princicpal Arterial")
    label = label.replace("princicpal", "principal")

    # synonyms / alternate spellings for the same canonical class
    SYNONYMS = {
        "interstate": "freeway",
        "local road": "local",
        # FL / NE labels that were falling through to blank K/D (see the WARNING this cell printed)
        "other principal arterial": "principal arterial",
        "primary arterial": "principal arterial",
        "principal arterials": "principal arterial",
        # bare "arterial" (FL) doesn't say principal vs minor; treated as principal arterial.
        # Change to "minor arterial" if that's what these segments actually are.
        "arterial": "principal arterial",
    }
    label = SYNONYMS.get(label, label)

    return label

cap_base = read_csv_any_encoding(CAP_BASE_PATH)
cap_base["_fc"] = cap_base["Facility Type"].apply(normalize_functional_class)
cap_base_lookup = cap_base.set_index("_fc")["Base Capacity (veh/hr/lane)"].to_dict()

master_df["through_lanes"] = pd.to_numeric(master_df["through_lanes"], errors="coerce")

# --- Align state-specific area_type / project_type labeling conventions ---

KY_AREA_TYPE_MAP = {
    "large urbanized area": "urban",
    "small urbanized area": "urban",
    "small urban area": "suburban",
    "rural area": "rural",
}

def normalize_area_type(value):
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    return KY_AREA_TYPE_MAP.get(label, label)

master_df["area_type_normalized"] = master_df["area_type"].apply(normalize_area_type)


# ---------------------------------------------------------------------------
# K / D factors, joined on state + area type + functional class + year
# ---------------------------------------------------------------------------
kd_factors = read_csv_any_encoding(K_D_FACTORS_PATH)
_kd_needed = ["Year", "State", "Area Type", "Functional Class", "K Factor", "D Factor"]
_kd_missing = [c for c in _kd_needed if c not in kd_factors.columns]
if _kd_missing:
    raise KeyError(f"{os.path.basename(K_D_FACTORS_PATH)} is missing columns {_kd_missing}; found {list(kd_factors.columns)}")

kd_factors["_state"] = kd_factors["State"].astype(str).str.strip().str.upper()
kd_factors["_area"] = kd_factors["Area Type"].astype(str).str.strip().str.lower()
kd_factors["_fc"] = kd_factors["Functional Class"].apply(normalize_functional_class)
kd_factors["_year"] = pd.to_numeric(kd_factors["Year"], errors="coerce").astype("Int64")
KD_KEYS = ["_state", "_area", "_fc", "_year"]
_dupes = kd_factors.duplicated(KD_KEYS).sum()
if _dupes:
    print(f"WARNING: {_dupes} duplicate state/area/class/year rows in the K/D table; keeping the first of each")
kd_lookup = (kd_factors.drop_duplicates(KD_KEYS)
             .rename(columns={"K Factor": "k_factor", "D Factor": "d_factor", "Year Match": "kd_year_match"}))
kd_lookup = kd_lookup[KD_KEYS + [c for c in ["k_factor", "d_factor", "kd_year_match"] if c in kd_lookup.columns]]

# Sanity check: the K/D table and Cap_Base_Values.csv should normalize to the same canonical classes.
kd_classes = set(kd_lookup["_fc"].dropna())
cb_classes = set(cap_base_lookup.keys())
if kd_classes != CANONICAL_FUNCTIONAL_CLASSES or cb_classes != CANONICAL_FUNCTIONAL_CLASSES:
    print("WARNING: K/D table / Cap_Base_Values.csv classes don't match the expected "
          f"canonical set. K/D: {sorted(kd_classes)} | Cap_Base_Values: {sorted(cb_classes)} "
          f"| expected: {sorted(CANONICAL_FUNCTIONAL_CLASSES)}")

# --- functional class used for the K/D join and the capacity calculation ---
# Same as functional_class (normalized), except that blank / "unknown" classes are filled from the facility
# name where it is unambiguous: "I-75", "Interstate ..." and "... Expressway" are freeways, and a name containing
# "RAMP" is a ramp. ~30% of rows (mostly KY) have no functional class, and without this interstates such as I-75
# got no K/D (so no peak_hour_volume) and were run through the arterial capacity parameters. functional_class
# itself is unchanged in the output.
_FREEWAY_NAME = r"^\s*I\s*-?\s*\d+\b|INTERSTATE|EXPRESSWAY|EXPWY|FREEWAY"
_RAMP_NAME = r"\bRAMP\b"

def capacity_functional_class(df):
    fc = df["functional_class"].map(normalize_functional_class)
    missing = fc.isna() | ~fc.isin(CANONICAL_FUNCTIONAL_CLASSES)
    name = df["facility_name"].astype("string").fillna("") if "facility_name" in df.columns else pd.Series("", index=df.index)
    is_ramp_name = name.str.contains(_RAMP_NAME, case=False, regex=True)
    is_fwy_name = name.str.contains(_FREEWAY_NAME, case=False, regex=True)
    fc = fc.where(~(missing & is_ramp_name), "ramp")
    fc = fc.where(~(missing & ~is_ramp_name & is_fwy_name), "freeway")
    from_name = missing & (is_ramp_name | is_fwy_name)
    return fc.fillna("unknown"), from_name


# Area type: the K/D table only has Urban and Rural. area_type_normalized (above) is urban / suburban / rural / ...
# Edit this map if a label should go the other way (e.g. exurban -> rural).
KD_AREA_MAP = {"urban": "urban", "suburban": "urban", "exurban": "urban", "rural": "rural"}

# State: two-letter code. States not in the K/D table use its national ("US") rows.
_KD_STATE_NAMES = {"kentucky": "KY", "florida": "FL", "minnesota": "MN", "new york": "NY", "new hampshire": "NH",
                   "nebraska": "NE", "ohio": "OH", "california": "CA"}
def _kd_state(value):
    if pd.isna(value):
        return None
    v = str(value).strip()
    return v.upper() if len(v) == 2 else _KD_STATE_NAMES.get(v.lower(), v.upper())

_kd_states = set(kd_lookup["_state"])
_yr_min, _yr_max = int(kd_lookup["_year"].min()), int(kd_lookup["_year"].max())

_obs_year = pd.to_datetime(master_df["observation_date"], errors="coerce", format="mixed").dt.year
_keys = pd.DataFrame({
    "_state": master_df["state"].apply(_kd_state),
    "_area": master_df["area_type_normalized"].map(KD_AREA_MAP),
    # functional class, with blank/unknown filled from the facility name (I-75 -> freeway, "... RAMP ..." -> ramp)
    "_fc": capacity_functional_class(master_df)[0],
    # years outside the table (e.g. 2026 counts) use the nearest table year
    "_year": _obs_year.clip(_yr_min, _yr_max).astype("Int64"),
}, index=master_df.index)
_no_state = _keys["_state"].notna() & ~_keys["_state"].isin(_kd_states)
if _no_state.any():
    print(f"Note: states not in the K/D table use national (US) values: "
          f"{sorted(_keys.loc[_no_state, '_state'].unique())} ({_no_state.sum():,} rows)")
    _keys.loc[_no_state, "_state"] = "US"
_clamped = (_obs_year < _yr_min) | (_obs_year > _yr_max)
if _clamped.any():
    print(f"Note: {_clamped.sum():,} rows have observation years outside {_yr_min}-{_yr_max}; "
          f"they use the {_yr_min} / {_yr_max} K/D values.")

_kd_joined = _keys.merge(kd_lookup, on=KD_KEYS, how="left")
_kd_joined.index = master_df.index          # merge(how="left") keeps row order; restore master_df's index

# Rows whose functional class is still unknown (no class, and nothing in the facility name): use the average K and D
# of the surface-road classes (arterials + collectors) for the same state, area type and year. Without this, ~25% of
# rows (mostly KY) get no K/D and therefore no peak_hour_volume. Set to False to leave them blank.
KD_FILL_UNKNOWN_CLASS = True
KD_SURFACE_CLASSES = ["principal arterial", "minor arterial", "major collector", "minor collector"]
_unknown_fc = ~_keys["_fc"].isin(CANONICAL_FUNCTIONAL_CLASSES)
if KD_FILL_UNKNOWN_CLASS and _unknown_fc.any():
    _surface_kd = (kd_lookup[kd_lookup["_fc"].isin(KD_SURFACE_CLASSES)]
                   .groupby(["_state", "_area", "_year"])[["k_factor", "d_factor"]].mean().reset_index())
    _fill = _keys[["_state", "_area", "_year"]].merge(_surface_kd, on=["_state", "_area", "_year"], how="left")
    _fill.index = master_df.index
    for _col in ["k_factor", "d_factor"]:
        _kd_joined.loc[_unknown_fc, _col] = _fill.loc[_unknown_fc, _col]
    if "kd_year_match" in _kd_joined.columns:
        _kd_joined.loc[_unknown_fc & _fill["k_factor"].notna(), "kd_year_match"] = "Unknown class: surface-road average"
    print(f"K/D for {(_unknown_fc & _fill['k_factor'].notna()).sum():,} rows with unknown functional class "
          f"= average of {', '.join(KD_SURFACE_CLASSES)} (KD_FILL_UNKNOWN_CLASS)")

# Ramps are one-way, and their AADT is already one direction, so the directional split does not apply: D = 1.0.
# (The table's ramp D of 0.55-0.60 cut ramp peak-hour volumes by ~40-45%, while ramp capacity is for the whole ramp.)
_is_ramp = _keys["_fc"].eq("ramp")
_kd_joined.loc[_is_ramp, "d_factor"] = 1.0
if _is_ramp.any():
    print(f"D = 1.0 for {_is_ramp.sum():,} ramp rows (one-way)")

for _col in [c for c in ["kd_year_match", "k_factor", "d_factor"] if c in _kd_joined.columns]:
    master_df[_col] = _kd_joined[_col]

# Report why any rows came out blank
_blank = master_df["k_factor"].isna()
if _blank.any():
    _why = pd.Series(np.select(
        [_keys["_state"].isna(), _keys["_area"].isna(), _unknown_fc & (not KD_FILL_UNKNOWN_CLASS), _keys["_year"].isna()],
        ["no state", "area type not urban/rural (" + master_df["area_type"].astype(str) + ")",
         "functional class not recognized (" + master_df["functional_class"].astype(str) + ")", "no observation date"],
        default="no matching row in K/D table"), index=master_df.index)[_blank]
    print(f"{_blank.sum():,} of {len(master_df):,} rows have blank k_factor / d_factor:")
    print(_why.value_counts().to_string())
print(f"K/D matched for {(~_blank).sum():,} of {len(master_df):,} rows")

_unmatched = sorted(
    v for v in _keys["_fc"].dropna().unique()
    if v not in CANONICAL_FUNCTIONAL_CLASSES and v != "unknown"
)
if _unmatched:
    print(f"WARNING: functional_class values not recognized after normalization: {_unmatched} "
          "-- these rows will have blank k_factor/d_factor. Add them to normalize_functional_class.")

PROJECT_TYPE_CATEGORY_MAP = {
    # MN
    "road-capacity-expansion": "widening",
    "new-roadway": "new construction",
    # KY
    "major widen": "widening",
    "ichg reconst": "widening",
    "new road": "new construction",
    "new route": "new construction",
    "new interchange": "new construction",
}

def normalize_project_type(value):
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    return PROJECT_TYPE_CATEGORY_MAP.get(label, label)

master_df["project_type_category"] = master_df["project_type"].apply(normalize_project_type)

master_df[["state", "observation_date", "area_type", "area_type_normalized", "functional_class",
           "k_factor", "d_factor", "kd_year_match", "through_lanes", "project_type", "project_type_category"]].head(10)

K/D for 1,345 rows with unknown functional class = average of principal arterial, minor arterial, major collector, minor collector (KD_FILL_UNKNOWN_CLASS)
D = 1.0 for 527 ramp rows (one-way)
K/D matched for 6,681 of 6,681 rows


,state,observation_date,area_type,area_type_normalized,functional_class,k_factor,d_factor,kd_year_match,through_lanes,project_type,project_type_category
0,FL,1997-01-01,urban,urban,principal arterial,0.102,0.579,Carried back,4.0,capacity expansion,capacity expansion
1,FL,1998-01-01,urban,urban,principal arterial,0.102,0.579,Carried back,4.0,capacity expansion,capacity expansion
2,FL,1999-01-01,urban,urban,principal arterial,0.102,0.579,Carried back,4.0,capacity expansion,capacity expansion
3,FL,2000-01-01,urban,urban,principal arterial,0.102,0.579,Carried back,4.0,capacity expansion,capacity expansion
4,FL,2001-01-01,urban,urban,principal arterial,0.102,0.579,Carried back,4.0,capacity expansion,capacity expansion
5,FL,2010-01-01,urban,urban,principal arterial,0.102,0.579,In effect / measured,4.0,capacity expansion,capacity expansion
6,FL,2011-01-01,urban,urban,principal arterial,0.102,0.579,In effect / measured,4.0,capacity expansion,capacity expansion
7,FL,2012-01-01,urban,urban,principal arterial,0.090,0.579,In effect / measured,4.0,capacity expansion,capacity expansion
8,FL,2013-01-01,urban,urban,principal arterial,0.090,0.579,In effect / measured,4.0,capacity expansion,capacity expansion
9,FL,2014-01-01,urban,urban,principal arterial,0.090,0.579,In effect / measured,4.0,capacity expansion,capacity expansion


## 5. Add county-level socioeconomic data

Merges in columns from `us_county_socioeconomic_1995_2025.csv` (population, employment, avg annual pay, fuel prices, and their 2025-dollar equivalents) onto `master_df`, matched on **state + county + year**:

- `state` (abbreviation, e.g. `MN`) is matched whether the socioeconomic file stores full state names or abbreviations.
- `county` is normalized by stripping the 'County' suffixes and matched ignoring case and periods, so `"Ramsey"` matches `"Ramsey County"` and `"St. Johns"` matches `"ST JOHNS COUNTY"`.
- Combined-county POIs (e.g. `"Hennepin and Ramsey"`) are matched by averaging the socioeconomic values across each constituent county for that year.
- The year used for matching is the calendar year of `observation_date`.
- The socioeconomic file's own state, county, year (and any FIPS/ID) columns are **not** included in the output. Headers are lower-cased with underscores turned into spaces, so the output columns keep the names used downstream (`population`, `employment`, `avg annual pay`, `avg annual pay (2025$)`, `gasoline price`, ...). If the file names a key or required column differently, the cell stops and lists the columns it found; add the name to `KEY_ALIASES` / `VALUE_ALIASES`.
- Rows with no match (e.g. observation years outside 1995–2025) come out blank for these columns.

This runs right after `k_factor`/`d_factor` are added and before `peak_hour_volume`, so these columns land between `d_factor` and `peak_hour_volume` in the final `combined_output.csv`. It also has to run before the Post-Construction Capacity Expansion Equation below, since that equation's `Δln(Pop_r)` and `Δln(Inc_r)` terms are built from the `population` and `avg annual pay (2025$)` columns added here.

In [8]:
US_STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "District of Columbia": "DC",
}
_STATE_NAME_TO_ABBREV = {k.lower(): v for k, v in US_STATE_ABBREV.items()}
_VALID_ABBREVS = set(US_STATE_ABBREV.values())

COUNTY_SUFFIXES = [
    " County", " Parish", " Borough", " Census Area", " Municipality",
    " Municipio", " City and Borough", " city", " City", " Region",
]

def normalize_county(name):
    # Strip common county/parish/borough suffixes so names match across files.
    if pd.isna(name):
        return None
    name = str(name).strip()
    for suf in COUNTY_SUFFIXES:
        if name.lower().endswith(suf.lower()):
            return name[: -len(suf)].strip()
    return name

def county_key(name):
    """Case/punctuation-insensitive match key: 'St. Johns County' and 'ST JOHNS' -> 'st johns'."""
    name = normalize_county(name)
    if name is None:
        return None
    return re.sub(r"\s+", " ", name.replace(".", "").replace("'", "")).strip().lower()

def state_key(value):
    """Accept either a full state name ('Minnesota') or an abbreviation ('MN') and return 'MN'."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.upper() in _VALID_ABBREVS:
        return s.upper()
    return _STATE_NAME_TO_ABBREV.get(s.lower())

# ---------------------------------------------------------------------------
# Load the 1995-2025 county socioeconomic table
# ---------------------------------------------------------------------------
SOCIO_PATH = os.path.join(FACTOR_TABLES_ROOT, "us_county_socioeconomic_1995_2025.csv")

socio_df = read_csv_any_encoding(SOCIO_PATH, low_memory=False)

# Standardize headers: lower case, underscores -> spaces, single spaces.
# (The rest of the notebook and Exploratory_Data_Analysis.ipynb expect lower-case names such as
#  "population", "employment", "avg annual pay", "avg annual pay (2025$)", "gasoline price".)
socio_df.columns = [re.sub(r"\s+", " ", str(c).replace("_", " ")).strip().lower() for c in socio_df.columns]

# Key columns may be named differently in the new table; map the first one found onto state / county / year.
KEY_ALIASES = {
    "state":  ["state", "state name", "statename", "state abbrev", "state abbreviation", "st"],
    "county": ["county", "county name", "countyname", "area name"],
    "year":   ["year", "yr", "data year"],
}
for key, aliases in KEY_ALIASES.items():
    found = next((a for a in aliases if a in socio_df.columns), None)
    if found is None:
        raise KeyError(f"Couldn't find a '{key}' column in {os.path.basename(SOCIO_PATH)}. "
                       f"Columns are: {list(socio_df.columns)}. Add the right name to KEY_ALIASES['{key}'].")
    if found != key:
        socio_df = socio_df.rename(columns={found: key})

# Value columns that downstream code needs, with alternate spellings the new table might use.
VALUE_ALIASES = {
    "population":             ["pop", "total population", "population estimate"],
    "employment":             ["total employment", "employed"],
    "avg annual pay":         ["average annual pay", "avg annual pay (nominal)", "avg annual pay nominal"],
    "avg annual pay (2025$)": ["average annual pay (2025$)", "avg annual pay 2025$", "avg annual pay 2025",
                               "avg annual pay (2025 $)", "real avg annual pay"],
    "gasoline price":         ["gas price", "gasoline price (nominal)"],
}
for target, aliases in VALUE_ALIASES.items():
    if target not in socio_df.columns:
        found = next((a for a in aliases if a in socio_df.columns), None)
        if found:
            socio_df = socio_df.rename(columns={found: target})

socio_df["state_abbrev"] = socio_df["state"].apply(state_key)
socio_df["county_clean"] = socio_df["county"].apply(county_key)
socio_df["year"] = pd.to_numeric(socio_df["year"], errors="coerce").astype("Int64")

_bad_states = socio_df.loc[socio_df["state_abbrev"].isna(), "state"].dropna().unique()
if len(_bad_states):
    print(f"Note: {len(_bad_states)} state value(s) in the socioeconomic file weren't recognized and are skipped, "
          f"e.g. {list(_bad_states[:5])}")

# Value columns = everything except the keys and ID-like columns (FIPS codes etc.).
# Numbers stored as text ("477,819", "$32,080") are converted; purely text columns are dropped.
_EXCLUDE = {"state", "county", "year", "state_abbrev", "county_clean"}
SOCIO_VALUE_COLS = []
for c in socio_df.columns:
    if c in _EXCLUDE or "fips" in c or c.endswith(" id") or c == "geoid":
        continue
    if not pd.api.types.is_numeric_dtype(socio_df[c]):
        socio_df[c] = pd.to_numeric(
            socio_df[c].astype(str).str.replace(r"[,$\s]", "", regex=True), errors="coerce"
        )
    if socio_df[c].notna().any():
        SOCIO_VALUE_COLS.append(c)

_missing_needed = [c for c in ["population", "avg annual pay (2025$)"] if c not in SOCIO_VALUE_COLS]
if _missing_needed:
    raise KeyError(f"{os.path.basename(SOCIO_PATH)} is missing {_missing_needed}, which Section 7 uses for "
                   f"d_ln_pop / d_ln_income. Columns found: {SOCIO_VALUE_COLS}. Add the right names to VALUE_ALIASES.")
_missing_eda = [c for c in ["employment", "avg annual pay", "gasoline price"] if c not in SOCIO_VALUE_COLS]
if _missing_eda:
    print(f"Note: {_missing_eda} not found in the socioeconomic file; Table 2 in Exploratory_Data_Analysis.ipynb uses these.")

print(f"Socioeconomic table: {len(socio_df):,} rows, years "
      f"{socio_df['year'].min()}-{socio_df['year'].max()}, value columns: {SOCIO_VALUE_COLS}")

socio_lookup = (
    socio_df.dropna(subset=["state_abbrev", "county_clean", "year"])
    .groupby(["state_abbrev", "county_clean", "year"])[SOCIO_VALUE_COLS].mean()
)

# ---------------------------------------------------------------------------
# Match onto master_df by state + county + observation year
# ---------------------------------------------------------------------------
# so this cell can be re-run without duplicating columns
master_df = master_df.drop(columns=[c for c in SOCIO_VALUE_COLS if c in master_df.columns])

master_df["observation_year"] = pd.to_datetime(master_df["observation_date"], errors="coerce").dt.year
master_df["county"] = master_df["county"].apply(normalize_county)
master_df["state"] = master_df["state"].astype(str).str.strip().str.upper()

_COUNTY_SPLIT = re.compile(r"\s*(?:,|&|/|\band\b)\s*", re.IGNORECASE)

def lookup_socio(state, county, year):
    """Socioeconomic values for one state/county/year. Combined-county POIs
    (e.g. 'Hennepin and Ramsey') get the average across their constituent counties."""
    if pd.isna(county) or pd.isna(year):
        return pd.Series(np.nan, index=SOCIO_VALUE_COLS)
    parts = [county_key(p) for p in _COUNTY_SPLIT.split(str(county)) if p.strip()]
    hits = [socio_lookup.loc[(state, p, int(year))] for p in parts if (state, p, int(year)) in socio_lookup.index]
    if not hits:
        return pd.Series(np.nan, index=SOCIO_VALUE_COLS)
    return pd.concat(hits, axis=1).mean(axis=1)

# look up each distinct state/county/year once, then merge back onto every row
_keys = master_df[["state", "county", "observation_year"]].drop_duplicates()
_vals = _keys.apply(lambda r: lookup_socio(r["state"], r["county"], r["observation_year"]), axis=1)
_socio_keys = pd.concat([_keys.reset_index(drop=True), _vals.reset_index(drop=True)], axis=1)

socio_matched = master_df[["state", "county", "observation_year"]].merge(
    _socio_keys, on=["state", "county", "observation_year"], how="left"
)[SOCIO_VALUE_COLS]
socio_matched.index = master_df.index

_no_match = socio_matched.isna().all(axis=1)
if _no_match.any():
    print(f"WARNING: {_no_match.sum()} row(s) had no socioeconomic match (check state/county/year):")
    print(master_df.loc[_no_match, ["state", "county", "observation_year"]].drop_duplicates()
          .sort_values(["state", "county", "observation_year"]).to_string(index=False))

master_df = pd.concat([master_df, socio_matched], axis=1).drop(columns=["observation_year"])

master_df[["state", "county", "d_factor"] + SOCIO_VALUE_COLS].head(10)

Socioeconomic table: 98,594 rows, years 1995-2025, value columns: ['population', 'employment', 'avg annual pay', 'gasoline price', 'diesel price', 'cpi adjustment factor (to 2025$)', 'avg annual pay (2025$)', 'gasoline price (2025$)', 'diesel price (2025$)']


,state,county,d_factor,population,employment,avg annual pay,gasoline price,diesel price,cpi adjustment factor (to 2025$),avg annual pay (2025$),gasoline price (2025$),diesel price (2025$)
0,FL,Brevard,0.579,459152.0,168517.0,28853.0,1.135689,1.229880,2.005875,57875.52,2.28,2.47
1,FL,Brevard,0.579,464716.0,174009.0,29540.0,0.954292,1.076145,1.975110,58344.76,1.88,2.13
2,FL,Brevard,0.579,468858.0,174299.0,30192.0,1.057089,1.145395,1.932431,58343.96,2.04,2.21
3,FL,Brevard,0.579,477819.0,180459.0,32080.0,1.369486,1.502725,1.869588,59976.37,2.56,2.81
4,FL,Brevard,0.579,486429.0,184725.0,32798.0,1.293817,1.419625,1.817860,59622.17,2.35,2.58
5,FL,Brevard,0.579,543990.0,188093.0,43770.0,2.620671,2.983914,1.476423,64623.06,3.87,4.41
6,FL,Brevard,0.579,544393.0,187114.0,44185.0,3.296276,3.839798,1.431246,63239.60,4.72,5.50
7,FL,Brevard,0.579,547029.0,187556.0,44153.0,3.369606,3.967562,1.402227,61912.55,4.72,5.56
8,FL,Brevard,0.579,550349.0,187497.0,44364.0,3.317914,3.920852,1.381985,61310.37,4.59,5.42
9,FL,Brevard,0.579,555669.0,189748.0,44484.0,3.178466,3.810948,1.359924,60494.87,4.32,5.18


## 6. Peak-hour volume

Adds `peak_hour_volume` = `AADT x K x D` (peak-direction design-hour volume), computed from the observed volume column (only where the unit is `AADT`), using the `k_factor` / `d_factor` from Section 4. On ramps D = 1.0, since ramp counts are already one-way. The cell prints how many rows were computed and why any are blank.

This is an **hourly** volume. HCM demand-to-capacity ratios use the peak 15-minute flow rate (volume ÷ PHF, default PHF 0.94), so a V/C built from this column runs about 6% lower than an HCM d/c ratio.

This runs after the socioeconomic merge (Section 5) so that, column-wise, the socioeconomic fields sit between `d_factor` and `peak_hour_volume` in `combined_output.csv`.


In [9]:
# Peak-direction design-hour volume: AADT x K x D (K = design-hour share of AADT, D = peak-direction share).
# AADT here is the two-way count except on ramps, where it is one-way and D = 1.0 (Section 4).
# This is an hourly volume. HCM demand-to-capacity ratios use the peak 15-minute flow rate, V / PHF
# (HCM 7.1 Eq. 13-1; default PHF 0.94), so V/C computed from this column is ~6% lower than an HCM d/c.
_is_aadt = master_df["observation_unit"].astype(str).str.strip().str.upper().eq("AADT")
_aadt = pd.to_numeric(master_df["observation_value"], errors="coerce")
master_df["peak_hour_volume"] = np.where(
    _is_aadt,
    _aadt * master_df["k_factor"] * master_df["d_factor"],
    np.nan,
)
_phv_blank = master_df["peak_hour_volume"].isna()
print(f"peak_hour_volume computed for {(~_phv_blank).sum():,} of {len(master_df):,} rows")
if _phv_blank.any():
    print(pd.Series(np.select(
        [~_is_aadt, _aadt.isna(), master_df["k_factor"].isna()],
        ["observation_unit is not AADT", "observation_value not numeric", "no K/D factor"],
        default="other"), index=master_df.index)[_phv_blank].value_counts().to_string())

master_df[[
    "functional_class", "through_lanes", "k_factor", "d_factor",
    "population", "avg annual pay (2025$)",
    "observation_value", "peak_hour_volume",
]].head(10)


peak_hour_volume computed for 6,681 of 6,681 rows


,functional_class,through_lanes,k_factor,d_factor,population,avg annual pay (2025$),observation_value,peak_hour_volume
0,principal arterial,4.0,0.102,0.579,459152.0,57875.52,29800,1759.9284
1,principal arterial,4.0,0.102,0.579,464716.0,58344.76,30500,1801.2690
2,principal arterial,4.0,0.102,0.579,468858.0,58343.96,30100,1777.6458
3,principal arterial,4.0,0.102,0.579,477819.0,59976.37,30500,1801.2690
4,principal arterial,4.0,0.102,0.579,486429.0,59622.17,30500,1801.2690
5,principal arterial,4.0,0.102,0.579,543990.0,64623.06,27000,1594.5660
6,principal arterial,4.0,0.102,0.579,544393.0,63239.60,27000,1594.5660
7,principal arterial,4.0,0.090,0.579,547029.0,61912.55,22500,1172.4750
8,principal arterial,4.0,0.090,0.579,550349.0,61310.37,26000,1354.8600
9,principal arterial,4.0,0.090,0.579,555669.0,60494.87,25500,1328.8050


## 7. Post-Construction Capacity Expansion Equation

Replaces the old flat `Cap_base x N x f_hv x PHF x f_p x f_g` lookup with the full model from
*Proposed Post-Capacity Expansion Equation (rev2)*:

$$\text{PostCapacity}_i = C_{eng,i} \times \exp\big(\beta_{LM}\Delta LM_i + \beta_L \Delta L_i + \beta_{LT}\Delta LT_i + \beta_{RT}\Delta RT_i + \beta_W W_i + \beta_E E_i + \beta_S S_i + \beta_F F_i + \beta_A \Delta\ln(AADT_i) + \beta_P \Delta\ln(Pop_r) + \beta_Y \Delta\ln(Inc_r) + \beta_{SL} SL_i + \beta_{SIG} SIG_i + \beta_{AT} AT_i\big)$$

- Before rows: $C_{eng,pre} \times e^{\beta_{SIG}SIG_{pre} + \beta_{AT}AT}$. 
- After rows: $C_{eng,post} \times e^{\text{change terms} + \beta_{SIG}SIG_{post} + \beta_{AT}AT}$.

**$C_{eng}$ (engineering baseline)** is computed with the SPR20-597 HERS free-flow-speed method (Section 3
of the doc): FFS → lane-width penalty → signal adjustment → stop-sign adjustment → per-lane capacity → x lanes x
heavy-vehicle factor, using HCM 7.1 for the capacity step:

- **Roads with signals or stop control**: per-lane flow where the HCM basic-segment speed-flow curve, run with the
  HERS-reduced FFS, reaches **35 pc/mi/ln** (HCM 7.1 Equation 13-15 with no weaving, solved in closed form). This is
  what the doc's Section 3 describes; the notebook previously approximated it as 0.9 x FFS x 35. The 35 pc/mi/ln value
  comes from HCM Chapter 13 (weaving) and Chapter 14 (merge/diverge), where it is the breakdown density for turbulent
  segments; here it stands in for the friction from signals and stops. The result is capped at the uncontrolled
  capacity of the same road, so a signal can only lower capacity.
- **Uncontrolled roads**: HCM basic-segment capacity, reached at **45 pc/mi/ln** (Ch. 13 Step 4): freeways from
  Exhibit 14-8 (2,250-2,400 pc/h/ln), multilane roads from Exhibit 14-9 (1,900-2,200; previously allowed up to 2,300),
  one-lane ramps from Exhibit 14-10 (1,800-2,200), and 1,700 pc/h for one lane per direction.
- **Heavy vehicles**: $f_{HV} = 1/(1 + P_T(E_T - 1))$ with the HCM 7.1 default truck shares (5% urban, 12% rural) and
  $E_T$ = 2.0 for level terrain, giving 0.952 urban and 0.893 rural. Set `USE_HCM_DEFAULT_FHV = False` to use the single
  value from `Cap_Cal_Values.csv` instead.

It uses each POI's **post-construction** geometry for after rows and pre-construction geometry for before rows.
**Changes made to fit the project data** (see the comments in the next three cells; none of them change the
columns of `combined_output.csv`):

1. **Functional class for capacity**: blank / `unknown` classes (~30% of rows, mostly KY) are filled from the facility
   name when it is unambiguous: `I-75`, `Interstate ...`, `... Expressway` → freeway; names containing `RAMP` → ramp.
   Interstates were previously run through the arterial (interrupted-flow) parameters.
2. **Signals per mile**: KY records no signal data and NY/NH only some. A blank was treated as 0 signals, so every
   KY arterial was capacity-rated as an uncontrolled multilane highway. Blanks are now filled with the median signal
   density of POIs with data in the same functional class and urban/rural group: 1 signal/mile for urban arterials,
   0 for rural roads, freeways and ramps.
3. **Stop signs**: the HERS stop-sign delay is applied only to collectors and local roads. On arterials the recorded
   stop signs are side-street stops and don't reduce the arterial's capacity.
4. **Lane width**: a blank lane width means no lane-width penalty (it previously got the maximum, 6.5 mph).
5. **Speed limit**: blanks are filled with the median of POIs in the same state + class + urban/rural (then class +
   urban/rural), so POIs with lane counts but no speed limit get a capacity instead of a blank.
6. **Ramps**: a ramp with no lane count (or 0) is one lane; KY ramp stations, whose cards carry the mainline's lanes
   and speed, are also one lane.
7. **Functional class improvement (F)**: now 1 when the POI moves up the hierarchy between before and after
   (e.g. collector → arterial) instead of always 0.
8. **Area type**: POIs with no after counts use their before-period area type for the AT term instead of rural.

Demand terms ($\beta_A$, $\beta_P$, $\beta_Y$) and the terms already inside $C_{eng}$ ($\beta_{LM}$, $\beta_L$, $\beta_W$,
$\beta_{SL}$, $\beta_{SIG}$) stay out of the multiplier: `peak_hour_capacity` is compared with `peak_hour_volume`, so
building AADT growth into capacity would make the volume/capacity ratio partly circular.


In [10]:
# Load calibration betas (and HCM scalars) from Cap_Cal_Values.csv.

cap_cal = pd.read_csv(CAP_CAL_PATH, encoding="cp1252")

PARAM_TO_KEY = {
    "Heavy Vehicle Factor": "f_hv",
    "Peak Hour Factor": "PHF",
    "Driver Population Factor": "f_p",
    "Grade Adjustment Factor": "f_g",
    "Lanemiles Elasticity": "BLM",
    "Lane Count Effect": "BL",
    "Left Turn Lane Effect": "BLT",
    "Right Turn Lane Effect": "BRT",
    "Lane Width Effect": "BW",
    "Managed Lane Effect": "BE",
    "Shoulder Widening Effect": "BS",
    "Functional Class Effect": "BF",
    "Speed Limit Effect": "BSL",
    "Signals per Mile Effect": "BSIG",
    "Area Type Effect": "BAT",
    "Local Demand Change": "BA",
    "Population Growth": "BP",
    "Income Growth": "BY",
}
cap_cal["_key"] = cap_cal["Parameter"].str.strip().map(PARAM_TO_KEY)
missing = cap_cal[cap_cal["_key"].isna()]
if len(missing):
    print("WARNING: unrecognized rows in Cap_Cal_Values.csv (not used):", missing["Parameter"].tolist())

cap_cal["_value"] = pd.to_numeric(cap_cal["Default Value"], errors="coerce")

bad_parse = cap_cal[cap_cal["_value"].isna()]
if len(bad_parse):
    print("WARNING: could not parse Default Value for:", bad_parse["Parameter"].tolist())

betas = cap_cal.dropna(subset=["_key"]).set_index("_key")["_value"].to_dict()

# Sanity check against the doc's Section 5 priors
for sym in ["BSIG", "BAT"]:
    if sym in betas and betas[sym] > 0:
        print(f"WARNING: {sym} = {betas[sym]} parsed as positive but the doc expects it negative "
              f"(see Proposed_Post-Capacity_Expansion_Equation_rev2, Section 5) - check Cap_Cal_Values.csv.")

f_hv = betas["f_hv"]  # heavy-vehicle factor, used in the Ceng calc below

betas

{'f_hv': 0.95,
 'PHF': 0.92,
 'f_p': 0.95,
 'f_g': 1.0,
 'BLM': 0.45,
 'BL': 0.35,
 'BLT': 0.1,
 'BRT': 0.06,
 'BW': 0.05,
 'BE': 0.15,
 'BS': 0.05,
 'BF': 0.1,
 'BSL': 0.05,
 'BSIG': -0.1,
 'BAT': -0.1,
 'BA': 1.0,
 'BP': 0.25,
 'BY': 0.25}

In [11]:
# Per-POI before/after geometry, used both for Ceng (post-construction geometry)
# and for the Delta/indicator variables in the exponential multiplier.
# A POI is keyed by project_id AND poi_id: the same count station can belong to several projects (55 stations do),
# each with its own open date and geometry, so grouping by poi_id alone mixed different projects together.
POI_KEY = ["project_id", "poi_id"]
GEOM_COLS = [
    "through_lanes", "lane_width", "left_turn_lanes", "right_turn_lanes",
    "shoulder_width", "express_hov_hot_lanes", "speed_limit",
    "signals_per_mile", "stop_signs_per_mile", "segment_length_miles",
]
for c in GEOM_COLS:
    master_df[c] = pd.to_numeric(master_df[c], errors="coerce")

geom_by_period = (
    master_df.groupby(POI_KEY + ["before_or_after"])[GEOM_COLS].mean().unstack("before_or_after")
)

def pre(col):
    return geom_by_period[col]["before"] if "before" in geom_by_period[col] else np.nan

def post(col):
    return geom_by_period[col]["after"] if "after" in geom_by_period[col] else np.nan

# capacity_functional_class() is defined in Section 4 (shared with the K/D join)
master_df["_fc_capacity"], master_df["_fc_from_name"] = capacity_functional_class(master_df)
print(f"Capacity functional class filled from facility name for "
      f"{master_df.drop_duplicates(POI_KEY)['_fc_from_name'].sum()} POIs "
      f"({master_df['_fc_from_name'].sum():,} rows); see capacity_functional_class()")

def _mode(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else None

# post-construction functional class / area type, used for Ceng flow-type and AT level
# (area_type_normalized, from Section 4, so KY's FHWA area-type labels line up with MN's)
_after_rows = master_df[master_df["before_or_after"] == "after"]
_before_rows = master_df[master_df["before_or_after"] == "before"]
fc_post = _after_rows.groupby(POI_KEY)["_fc_capacity"].agg(_mode)
fc_pre = _before_rows.groupby(POI_KEY)["_fc_capacity"].agg(_mode)
# area type: the after-period label, or the before-period label for POIs with no after counts
at_post = _after_rows.groupby(POI_KEY)["area_type_normalized"].agg(_mode)
at_any = master_df.groupby(POI_KEY)["area_type_normalized"].agg(_mode)
fc_from_name = master_df.groupby(POI_KEY)["_fc_from_name"].any()
poi_state = master_df.groupby(POI_KEY)["state"].agg(_mode)

poi = pd.DataFrame(index=geom_by_period.index)

# --- geometry deltas (post - pre / pre) ---
# d_lane_miles is multiplied by BLM, the lane-miles ELASTICITY, so it is the log change
# ln(post / pre), not the percent change (post - pre) / pre.
lane_miles_pre = pre("through_lanes") * pre("segment_length_miles")
lane_miles_post = post("through_lanes") * post("segment_length_miles")
d_lane_miles_full = np.log(lane_miles_post / lane_miles_pre)
d_lane_miles_lanes_only = np.log(post("through_lanes") / pre("through_lanes"))
poi["d_lane_miles"] = d_lane_miles_full.where(d_lane_miles_full.notna(), d_lane_miles_lanes_only)
poi["d_through_lanes"] = post("through_lanes") - pre("through_lanes")
poi["d_left_turn_lanes"] = post("left_turn_lanes") - pre("left_turn_lanes")
poi["d_right_turn_lanes"] = post("right_turn_lanes") - pre("right_turn_lanes")

# --- 0/1 indicators (W, E, S, SL) ---
poi["lane_width_improved"] = (post("lane_width") > pre("lane_width")).astype("Int64")
poi["express_lane_added"] = (post("express_hov_hot_lanes") > pre("express_hov_hot_lanes")).astype("Int64")
poi["shoulder_widened"] = (post("shoulder_width") > pre("shoulder_width")).astype("Int64")
poi["speed_limit_increased"] = (post("speed_limit") > pre("speed_limit")).astype("Int64")


# --- levels used directly (SIG, AT), from post-construction state ---
poi["signals_per_mile_post"] = post("signals_per_mile")   # blanks filled below
poi["stop_signs_per_mile_post"] = post("stop_signs_per_mile")   # blanks filled below
poi["through_lanes_post"] = post("through_lanes")
poi["lane_width_post"] = post("lane_width")
poi["speed_limit_post"] = post("speed_limit")
poi["functional_class_post"] = fc_post

# --- pre-construction geometry, used for the pre-construction Ceng (Section 7 base capacity).
# Any pre value that's missing falls back to the post value (i.e. assume that attribute didn't change).
poi["through_lanes_pre"] = pre("through_lanes").fillna(poi["through_lanes_post"])
poi["lane_width_pre"] = pre("lane_width").fillna(poi["lane_width_post"])
poi["speed_limit_pre"] = pre("speed_limit").fillna(poi["speed_limit_post"])
poi["signals_per_mile_pre"] = pre("signals_per_mile").fillna(poi["signals_per_mile_post"])
poi["stop_signs_per_mile_pre"] = pre("stop_signs_per_mile").fillna(poi["stop_signs_per_mile_post"])
poi["functional_class_pre"] = fc_pre.reindex(poi.index).fillna(poi["functional_class_post"])
poi["area_type_post"] = at_post.reindex(poi.index).fillna(at_any.reindex(poi.index))
poi["area_type_level"] = poi["area_type_post"].astype(str).str.strip().str.lower().isin(["urban", "suburban"]).astype(int)

# --- functional class improvement (F): 1 if the POI moved UP the functional hierarchy (e.g. collector -> arterial).
# A change from unknown to a known class is not an improvement.
FC_RANK = {"local": 1, "minor collector": 2, "major collector": 3, "minor arterial": 4,
           "principal arterial": 5, "freeway": 6}
_rank_pre = poi["functional_class_pre"].map(FC_RANK)
_rank_post = poi["functional_class_post"].map(FC_RANK)
poi["functional_class_improved"] = (_rank_post > _rank_pre).astype(int)   # NaN comparisons -> 0

# ---------------------------------------------------------------------------
# Fill gaps in the inputs the capacity formula needs (used for capacity only; the observation rows and
# combined_output.csv keep the values as recorded). Every fill is counted and printed.
# ---------------------------------------------------------------------------
poi["state"] = poi_state.reindex(poi.index)
poi["_urban"] = poi["area_type_level"].map({1: "urban", 0: "rural"})
_UNINTERRUPTED = {"freeway", "ramp"}
_fill_log = {}

def _group_fill(target_cols, groupings, stat):
    """Fill NaNs in each of target_cols from statistics of the known values (pre and post pooled),
    trying each grouping in turn (most specific first)."""
    known = pd.concat([
        poi[["state", "_urban"]].assign(fc=poi["functional_class_pre"], v=poi[c]) if c.endswith("_pre")
        else poi[["state", "_urban"]].assign(fc=poi["functional_class_post"], v=poi[c])
        for c in target_cols
    ]).dropna(subset=["v"])
    for c in target_cols:
        fc_col = poi["functional_class_pre"] if c.endswith("_pre") else poi["functional_class_post"]
        frame = poi[["state", "_urban"]].assign(fc=fc_col)
        before = poi[c].isna().sum()
        for keys in groupings:
            if not poi[c].isna().any():
                break
            ref = known.groupby(keys)["v"].agg(["count", stat])
            ref = ref.loc[ref["count"] >= 5, stat]
            lookup = frame[keys].merge(ref.rename("_fill"), left_on=keys, right_index=True, how="left")["_fill"]
            poi[c] = poi[c].fillna(pd.Series(lookup.values, index=poi.index))
        _fill_log[c] = before - poi[c].isna().sum()

# Speed limit: median of POIs with the same state + class + urban/rural, else class + urban/rural across states.
_group_fill(["speed_limit_pre", "speed_limit_post"],
            [["state", "fc", "_urban"], ["fc", "_urban"], ["fc"]], "median")

# Signals per mile: KY records none and NY/NH only some. Treating a blank as 0 ran every KY arterial as an
# uncontrolled multilane highway (~1,800-2,100 veh/h/ln). Fill blanks with the MEDIAN signal density of POIs with
# data in the same class + urban/rural (FL/MN/NE/NY): in this data that is 1 signal/mile for urban arterials and
# 0 for rural roads. The median (the typical POI) is used rather than the mean, because capacity changes sharply
# between 0 and >0 signals and a fractional mean would put every rural road on the signalized formula.
# Freeways and ramps have no signals by definition.
for _c, _fc in [("signals_per_mile_pre", "functional_class_pre"), ("signals_per_mile_post", "functional_class_post")]:
    _mask = poi[_c].isna() & poi[_fc].isin(_UNINTERRUPTED)
    poi.loc[_mask, _c] = 0
# surface roads of unknown class (or a class with < 5 POIs with data) use the pooled arterial/collector median
_sig_known = pd.concat([
    poi[["_urban"]].assign(fc=poi["functional_class_post"], v=poi["signals_per_mile_post"]),
    poi[["_urban"]].assign(fc=poi["functional_class_pre"], v=poi["signals_per_mile_pre"]),
]).dropna(subset=["v"])
_sig_by_class = _sig_known.groupby(["fc", "_urban"])["v"].agg(["count", "median"])
_sig_by_class = _sig_by_class.loc[_sig_by_class["count"] >= 5, "median"]
_sig_surface = _sig_known[~_sig_known["fc"].isin(_UNINTERRUPTED)].groupby("_urban")["v"].median()
for _c, _fc in [("signals_per_mile_pre", "functional_class_pre"), ("signals_per_mile_post", "functional_class_post")]:
    _before = poi[_c].isna().sum()
    _key = pd.MultiIndex.from_arrays([poi[_fc], poi["_urban"]])
    _by_class = pd.Series(_sig_by_class.reindex(_key).values, index=poi.index)
    _by_surface = poi["_urban"].map(_sig_surface)
    poi[_c] = poi[_c].fillna(_by_class).fillna(_by_surface)
    _fill_log[_c] = _before - poi[_c].isna().sum()
print("Signal density used to fill blanks (signals/mile, median of POIs with data):")
print(_sig_by_class.round(2).to_string())
print("  other surface roads:", _sig_surface.round(2).to_dict())

# Stop signs: blank -> 0 (only used for collectors / local roads, see hers_ceng in the next cell)
for _c in ["stop_signs_per_mile_pre", "stop_signs_per_mile_post"]:
    _fill_log[_c] = poi[_c].isna().sum()
    poi[_c] = poi[_c].fillna(0)

# Ramps: a ramp with no lane count (or 0) is one lane. Ramps whose class came from the facility name (KY
# "I-64 RAMP from KY 1848") carry the MAINLINE's lane count and speed in the card, so they are also set to 1 lane.
for _c, _fc in [("through_lanes_pre", "functional_class_pre"), ("through_lanes_post", "functional_class_post")]:
    _mask = (poi[_fc] == "ramp") & (poi[_c].isna() | (poi[_c] == 0) | fc_from_name.reindex(poi.index).fillna(False))
    _fill_log[_c + " (ramps -> 1 lane)"] = int(_mask.sum())
    poi.loc[_mask, _c] = 1

print("POI-periods filled for the capacity calculation:")
print(pd.Series(_fill_log).to_string())

# --- demand delta: mean AADT pre vs post per POI ---
master_df["observation_value"] = pd.to_numeric(master_df["observation_value"], errors="coerce")
aadt_obs = master_df[master_df["observation_unit"] == "AADT"]
aadt_by_period = aadt_obs.groupby(POI_KEY + ["before_or_after"])["observation_value"].mean().unstack("before_or_after")
poi["d_ln_aadt"] = np.log(aadt_by_period["after"] / aadt_by_period["before"])

# --- regional population & income growth: pre vs post, straight from the county
# socioeconomic merge (Section 5) for each POI's own state/county/year ---
SOCIO_GROWTH_COLS = ["population", "avg annual pay (2025$)"]
socio_by_period = (
    master_df.groupby(POI_KEY + ["before_or_after"])[SOCIO_GROWTH_COLS].mean().unstack("before_or_after")
)

def socio_pre(col):
    return socio_by_period[col]["before"] if "before" in socio_by_period[col] else np.nan

def socio_post(col):
    return socio_by_period[col]["after"] if "after" in socio_by_period[col] else np.nan

poi["d_ln_pop"] = np.log(socio_post("population") / socio_pre("population"))
poi["d_ln_income"] = np.log(socio_post("avg annual pay (2025$)") / socio_pre("avg annual pay (2025$)"))

poi.head(10)

Capacity functional class filled from facility name for 19 POIs (193 rows); see capacity_functional_class()
Signal density used to fill blanks (signals/mile, median of POIs with data):
fc                  _urban
freeway             rural     0.0
                    urban     0.0
minor arterial      rural     0.0
                    urban     1.0
minor collector     rural     0.0
principal arterial  rural     0.0
                    urban     1.0
ramp                rural     0.0
                    urban     0.0
  other surface roads: {'rural': 0.0, 'urban': 1.0}
POI-periods filled for the capacity calculation:
speed_limit_pre                         101
speed_limit_post                        100
signals_per_mile_pre                    301
signals_per_mile_post                   325
stop_signs_per_mile_pre                 359
stop_signs_per_mile_post                382
through_lanes_pre (ramps -> 1 lane)      29
through_lanes_post (ramps -> 1 lane)     29


d_lane_miles  d_through_lanes  d_left_turn_lanes  d_right_turn_lanes  lane_width_improved  express_lane_added  shoulder_widened  speed_limit_increased  signals_per_mile_post  \
project_id poi_id                                                                                                                                                                                  
07-366.00  034D64      0.405465              2.0                NaN                 NaN                    0                   0                 0                      0                    0.0   
           034D66           NaN              NaN                NaN                 NaN                    0                   0                 0                      0                    1.0   
1-151.00   111054      0.693147              2.0                NaN                 NaN                    0                   0                 0                      0                    0.0   
           111786      0.693147              2.0                NaN                 NaN                    0                   0                 0                      0                    0.0   
1-180.10   111502      0.693147              2.0                NaN                 NaN                    1                   0                 0                      0                    0.0   
           111542      0.693147              2.0                0.0                 0.0                    1                   0                 0                      0                    0.0   
1-180.11   111500      0.693147              2.0                NaN                 NaN                    1                   0                 0                      0                    0.0   
           111502      0.693147              2.0                NaN                 NaN                    1                   0                 0                      0                    0.0   
           111547      0.693147              2.0                NaN                 NaN                    1                   0                 0                      0                    0.0   
           111786      0.693147              2.0                NaN                 NaN                    0                   0                 0                      0                    0.0   

                   stop_signs_per_mile_post  through_lanes_post  lane_width_post  speed_limit_post functional_class_post  through_lanes_pre  lane_width_pre  speed_limit_pre  signals_per_mile_pre  \
project_id poi_id                                                                                                                                                                                    
07-366.00  034D64                       0.0                 6.0             12.0              55.0               freeway                4.0            12.0             55.0                   0.0   
           034D66                       0.0                 NaN              NaN               NaN                   NaN                4.0            12.0             55.0                   1.0   
1-151.00   111054                       0.0                 4.0             12.0              55.0    principal arterial                2.0            12.0             55.0                   0.0   
           111786                       0.0                 4.0             12.0              55.0    principal arterial                2.0            12.0             55.0                   0.0   
1-180.10   111502                       0.0                 4.0             12.0              55.0    principal arterial                2.0            11.0             55.0                   0.0   
           111542                       0.0                 4.0             12.0              55.0    principal arterial                2.0            11.0             55.0                   0.0   
1-180.11   111500                       0.0                 4.0             12.0  

In [12]:
# SPR20-597 HERS free-flow-speed method -> engineering baseline capacity (Ceng), Sections 2-3 of the doc,
# with the capacity values and speed-flow curve taken from HCM 7.1 (Chapters 13 and 14 and the Chapter 12 values they cite).
# Uses each POI's POST-construction geometry, since Ceng is defined for post-construction geometry.
UNINTERRUPTED_CLASSES = {"freeway", "ramp"}
STOP_CONTROLLED_CLASSES = {"major collector", "minor collector", "local"}
ONE_WAY_CLASSES = {"ramp"}  # lane counts on these are already one direction

# through_lanes in the project files is the two-way total. Set to False if yours are per direction.
THROUGH_LANES_ARE_TWO_WAY = True

# Heavy vehicles (HCM 7.1 Ch. 13/14 Exhibits 13-9 and 14-4 defaults: 5% urban, 12% rural; Ch. 12 level-terrain
# E_T = 2.0, f_HV = 1 / (1 + P_T (E_T - 1)) -> 0.952 urban, 0.893 rural). Set USE_HCM_DEFAULT_FHV = False to use the
# single "Heavy Vehicle Factor" from Cap_Cal_Values.csv for every POI instead.
USE_HCM_DEFAULT_FHV = True
HCM_DEFAULT_TRUCK_SHARE = {"urban": 0.05, "rural": 0.12}
HCM_ET_LEVEL = 2.0


def hcm_basic_capacity(ffs, kind):
    """Basic-segment capacity (pc/h/ln) and speed-flow breakpoint (pc/h/ln) by free-flow speed.
    Freeways: HCM 7.1 Exhibit 14-8 (2,400 at FFS >= 70 ... 2,250 at 55; = 2,200 + 10(FFS - 50)), breakpoint
    1,000 + 40(75 - FFS) (HCM Ch. 12; 1,400 at 65 mi/h as in the Ch. 13 examples).
    Multilane / surface roads: Exhibit 14-9 (2,200 at FFS >= 60 ... 1,900 at 45; = 1,900 + 20(FFS - 45)), breakpoint 1,400.
    Below the HCM range (FFS < 45 multilane, < 50 freeway) the linear relationships are extended down to 1,700."""
    ffs = np.asarray(ffs, dtype=float)
    fwy = np.asarray(kind) == "freeway"
    cap = np.where(fwy, np.clip(2200 + 10 * (ffs - 50), 1700, 2400), np.clip(1900 + 20 * (ffs - 45), 1700, 2200))
    bp = np.where(fwy, np.clip(1000 + 40 * (75 - ffs), 1000, None), 1400.0)
    return cap, bp


def hcm_flow_at_density(ffs, kind, density=35.0):
    """Per-lane flow (pc/h/ln) where the HCM basic-segment speed-flow curve reaches `density`.
    This is HCM 7.1 Equation 13-15 with no weaving (W = 0): v = density x S(v), where
    S(v) = FFS                                          for v <= BP
    S(v) = FFS - (FFS - c/45) ((v - BP) / (c - BP))^2   for BP < v <= c   (c is reached at 45 pc/mi/ln).
    Solved in closed form (the quadratic of Eq. 13-16 with W = 0)."""
    ffs = np.asarray(ffs, dtype=float)
    cap, bp = hcm_basic_capacity(ffs, kind)
    linear = density * ffs                                   # flow if speed is still FFS at this density
    k = (ffs - cap / 45) / (cap - bp) ** 2
    qa = density * k
    disc = 1 - 4 * qa * (bp - density * ffs)
    x = (-1 + np.sqrt(np.clip(disc, 0, None))) / (2 * np.where(qa > 0, qa, np.nan))
    curved = bp + x
    return np.where(linear <= bp, linear, np.minimum(curved, cap))


def hers_ceng(functional_class, speed_limit, lane_width, signals_per_mile, stop_signs_per_mile, through_lanes,
              area_type_level=None):
    fc_norm = functional_class.map(normalize_functional_class).fillna("unknown")
    is_uninterrupted = fc_norm.isin(UNINTERRUPTED_CLASSES)
    sigma = np.where(is_uninterrupted, 0.1427, 0.3907)
    beta_hers = np.where(is_uninterrupted, 0.2092, 0.18378)

    VSPLIM = speed_limit.astype(float)
    base_ffs = (np.exp(sigma**2 / 2) * VSPLIM**(-1/beta_hers))**(-beta_hers)

    # Lane-width penalty (12ft=0, 11ft=1.5mph, 10ft=4mph, <10ft=6.5mph).
    # A blank lane width is taken as standard (no penalty); before, NaN fell through to the 6.5 mph default.
    lw = lane_width.astype(float)
    fw = np.select([lw.isna() | (lw >= 12), lw >= 11, lw >= 10], [0, 1.5, 4], default=6.5)
    ffs_lw = base_ffs - fw

    nsig = signals_per_mile.astype(float).where(~is_uninterrupted, 0)
    zvdsig = np.where(nsig == 0, 0, 0.0687 * (1 - np.exp(-nsig / 24.4)))
    ffs_sig = 1 / (1/ffs_lw + zvdsig)

    # Stop signs: on arterials the recorded stop signs are side-street stops (the arterial itself does not stop),
    # so they don't cut the arterial's capacity. Only collectors / local roads take the HERS stop-sign delay.
    nstp = stop_signs_per_mile.astype(float).where(fc_norm.isin(STOP_CONTROLLED_CLASSES), 0).fillna(0)
    zvdstp = np.where(nstp == 0, 0, nstp * (1.9 + 0.067 * ffs_sig))
    ffs_final = 1 / (1/ffs_sig + zvdstp / 1000)

    # Signal / stop-controlled roads: flow where the HCM speed-flow curve (with the HERS-reduced FFS) reaches
    # 35 pc/mi/ln, per Section 3 of the doc ("use the HCM speed-density curve to find the speed corresponding to
    # 35 pc/mi/ln"). This replaces the old shortcut speed = 0.9 x FFS. 35 pc/mi/ln is the HCM's breakdown density
    # for TURBULENT segments (weaving, Ch. 13 Step 4; merge/diverge, Ch. 14), lower than the 45 pc/mi/ln of a basic
    # segment; it is used here as the stand-in for the friction that signals and stops add.
    s_cap = hcm_flow_at_density(ffs_final, np.where(fc_norm.eq("freeway"), "freeway", "multilane"), density=35.0)

    lanes = through_lanes.astype(float)
    if THROUGH_LANES_ARE_TWO_WAY:
        lanes = lanes.where(fc_norm.isin(ONE_WAY_CLASSES), lanes / 2)

    # Uninterrupted flow (no signals or stop signs): HCM basic-segment capacity, which is reached at 45 pc/mi/ln
    # (HCM 7.1 Ch. 13 Step 4: "basic segment breakdowns occur at a density of 45 pc/mi/ln").
    #   ramps:      Exhibit 14-10, one-lane ramp roadway capacity by ramp FFS (2,200 / 2,100 / 2,000 / 1,900 / 1,800)
    #   freeways:   Exhibit 14-8  (2,250-2,400 pc/h/ln)
    #   multilane:  Exhibit 14-9  (1,900-2,200 pc/h/ln; was capped at 2,300 before, above the HCM table)
    #   one lane per direction: 1,700 pc/h (HCM two-lane highway capacity per direction)
    no_control = (nsig.fillna(0) == 0) & (nstp == 0)
    ramp_cap = np.select([ffs_lw > 50, ffs_lw > 40, ffs_lw > 30, ffs_lw >= 20],
                         [2200, 2100, 2000, 1900], default=1800)
    freeway_cap, _ = hcm_basic_capacity(ffs_lw, np.full(len(ffs_lw), "freeway"))
    multilane_cap, _ = hcm_basic_capacity(ffs_lw, np.full(len(ffs_lw), "multilane"))
    hcm_cap = np.select(
        [fc_norm.eq("ramp"), fc_norm.eq("freeway"), lanes >= 2],
        [ramp_cap, freeway_cap, multilane_cap],
        default=1700,
    )
    # signals / stops can only lower capacity below the uncontrolled value for the same road
    s_cap = np.where(no_control, hcm_cap, np.minimum(s_cap, hcm_cap))

    if USE_HCM_DEFAULT_FHV and area_type_level is not None:
        p_t = np.where(np.asarray(area_type_level) == 1, HCM_DEFAULT_TRUCK_SHARE["urban"], HCM_DEFAULT_TRUCK_SHARE["rural"])
        fhv = 1 / (1 + p_t * (HCM_ET_LEVEL - 1))
    else:
        fhv = f_hv
    return s_cap * lanes * fhv


poi["engineering_capacity_ceng_pre"] = hers_ceng(
    poi["functional_class_pre"], poi["speed_limit_pre"], poi["lane_width_pre"],
    poi["signals_per_mile_pre"], poi["stop_signs_per_mile_pre"], poi["through_lanes_pre"],
    area_type_level=poi["area_type_level"],
)
poi["engineering_capacity_ceng_post"] = hers_ceng(
    poi["functional_class_post"], poi["speed_limit_post"], poi["lane_width_post"],
    poi["signals_per_mile_post"], poi["stop_signs_per_mile_post"], poi["through_lanes_post"],
    area_type_level=poi["area_type_level"],
)

poi[[
    "functional_class_post", "speed_limit_post", "through_lanes_pre", "through_lanes_post",
    "engineering_capacity_ceng_pre", "engineering_capacity_ceng_post",
]].head(10)

functional_class_post  speed_limit_post  through_lanes_pre  through_lanes_post  engineering_capacity_ceng_pre  engineering_capacity_ceng_post
project_id poi_id                                                                                                                                              
07-366.00  034D64               freeway              55.0                4.0                 6.0                    4283.485232                     6425.227848
           034D66                   NaN               NaN                4.0                 NaN                    3098.757339                             NaN
1-151.00   111054    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915
           111786    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915
1-180.10   111502    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915
           111542    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915
1-180.11   111500    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915
           111502    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915
           111547    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915
           111786    principal arterial              55.0                2.0                 4.0                    1517.857143                     3722.639915

In [13]:
CHANGE_TERMS = {
    "BLT": "d_left_turn_lanes",
    "BRT": "d_right_turn_lanes",
    "BE": "express_lane_added",
    "BS": "shoulder_widened",
    "BF": "functional_class_improved",
}

EXCLUDED_TERMS = {
    # already in Ceng (geometry)
    "BLM": "d_lane_miles",
    "BL": "d_through_lanes",
    "BW": "lane_width_improved",
    "BSL": "speed_limit_increased",
    # demand, not capacity
    "BA": "d_ln_aadt",
    "BP": "d_ln_pop",      # regional population growth, from us_county_socioeconomic_1995_2025.csv
    "BY": "d_ln_income",   # regional income growth, from us_county_socioeconomic_1995_2025.csv
}

_change_vars = poi[list(CHANGE_TERMS.values())].astype(float).replace([np.inf, -np.inf], np.nan)
_n_filled = _change_vars.isna().sum()
if _n_filled.any():
    print("Missing change terms treated as 0 (no change), POI counts:")
    print(_n_filled[_n_filled > 0].to_string())
_change_vars = _change_vars.fillna(0)

X_change = sum(betas[b] * _change_vars[col] for b, col in CHANGE_TERMS.items())
# BSIG (signals per mile) is left out: Ceng already applies the HERS signal-delay reduction,
# so the extra -10% per signal/mile counted signals twice.
X_level_pre = betas["BAT"] * poi["area_type_level"]
X_level_post = betas["BAT"] * poi["area_type_level"]

poi["capacity_multiplier_pre"] = np.exp(X_level_pre)
poi["capacity_multiplier"] = np.exp(X_change + X_level_post)
poi["peak_hour_capacity_pre"] = poi["engineering_capacity_ceng_pre"] * poi["capacity_multiplier_pre"]
poi["peak_hour_capacity"] = poi["engineering_capacity_ceng_post"] * poi["capacity_multiplier"]

# --- sanity check: flag capacities that are implausible per lane, and big lane-count jumps
_lanes_dir = np.where(
    THROUGH_LANES_ARE_TWO_WAY & ~poi["functional_class_post"].map(normalize_functional_class).isin(ONE_WAY_CLASSES),
    poi["through_lanes_post"] / 2, poi["through_lanes_post"],
)
poi["capacity_per_lane_post"] = poi["peak_hour_capacity"] / _lanes_dir
MAX_PLAUSIBLE_PER_LANE = 2400  # veh/hr/ln, roughly the HCM ceiling for a basic freeway segment
_suspect = poi[
    (poi["capacity_per_lane_post"] > MAX_PLAUSIBLE_PER_LANE)
    | ((poi["through_lanes_pre"] > 0) & (poi["through_lanes_post"] >= 3 * poi["through_lanes_pre"]))
]

# through_lanes = 0 (e.g. ramps with no lane count entered) would give a capacity of 0 -> leave blank instead
poi.loc[poi["through_lanes_pre"] == 0, "peak_hour_capacity_pre"] = np.nan
poi.loc[poi["through_lanes_post"] == 0, "peak_hour_capacity"] = np.nan

# Merge the POI-level results back onto every observation row, by project + POI AND period
_poi_out = pd.concat([
    poi[["engineering_capacity_ceng_pre", "capacity_multiplier_pre", "peak_hour_capacity_pre"]]
        .set_axis(["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], axis=1)
        .assign(before_or_after="before"),
    poi[["engineering_capacity_ceng_post", "capacity_multiplier", "peak_hour_capacity"]]
        .set_axis(["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], axis=1)
        .assign(before_or_after="after"),
]).reset_index()
master_df = master_df.drop(columns=["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], errors="ignore")
master_df = master_df.merge(_poi_out, on=POI_KEY + ["before_or_after"], how="left")

Missing change terms treated as 0 (no change), POI counts:
d_left_turn_lanes     396
d_right_turn_lanes    400


## 8. Spot-check the merged data

Displays a narrowed view of just the columns most relevant for spot-checking, the join keys, dates, before/after labels, and a few geometric attributes (lane counts, shoulder width), to visually verify the merges worked correctly before writing anything out.

In [14]:
master_df[[
    "project_id", "poi_id", "observation_date", "date_open_actual",
    "before_or_after", "before_or_after_binary", "years_from_open",
    "through_lanes", "right_turn_lanes", "shoulder_width",
    "state", "area_type", "functional_class", "k_factor", "d_factor", "kd_year_match",
    "population", "avg annual pay (2025$)",
    "peak_hour_volume", "peak_hour_capacity",
]]

,project_id,poi_id,observation_date,date_open_actual,before_or_after,before_or_after_binary,years_from_open,through_lanes,right_turn_lanes,shoulder_width,state,area_type,functional_class,k_factor,d_factor,kd_year_match,population,avg annual pay (2025$),peak_hour_volume,peak_hour_capacity
0,237506-2,701007,1997-01-01,2007-01-01,before,0,-10.0,4.0,0.0,6.0,FL,urban,principal arterial,0.102,0.579,Carried back,459152.0,57875.52,1759.9284,3455.062828
1,237506-2,701007,1998-01-01,2007-01-01,before,0,-9.0,4.0,0.0,6.0,FL,urban,principal arterial,0.102,0.579,Carried back,464716.0,58344.76,1801.2690,3455.062828
2,237506-2,701007,1999-01-01,2007-01-01,before,0,-8.0,4.0,0.0,6.0,FL,urban,principal arterial,0.102,0.579,Carried back,468858.0,58343.96,1777.6458,3455.062828
3,237506-2,701007,2000-01-01,2007-01-01,before,0,-7.0,4.0,0.0,6.0,FL,urban,principal arterial,0.102,0.579,Carried back,477819.0,59976.37,1801.2690,3455.062828
4,237506-2,701007,2001-01-01,2007-01-01,before,0,-6.0,4.0,0.0,6.0,FL,urban,principal arterial,0.102,0.579,Carried back,486429.0,59622.17,1801.2690,3455.062828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6676,20190293,50076,2021-01-01,NaT,after,1,NaN,7.0,0.0,8.0,NY,urban,freeway,0.100,0.550,Carried forward,2328520.0,74991.32,3795.5500,6481.780284
6677,20190293,50076,2022-01-01,NaT,after,1,NaN,7.0,0.0,8.0,NY,urban,freeway,0.100,0.550,Carried forward,2291363.0,71250.88,3930.4100,6481.780284
6678,20190293,50076,2023-01-01,NaT,after,1,NaN,7.0,0.0,8.0,NY,urban,freeway,0.100,0.550,Carried forward,2314747.0,70773.11,5279.5050,6481.780284
6679,20190293,50076,2024-01-01,NaT,after,1,NaN,7.0,0.0,8.0,NY,urban,freeway,0.100,0.550,Carried forward,2367034.0,69500.87,4966.0050,6481.780284


## 9. Write out the combined CSV

Adds a blank `comment` column (for manual notes) as the very last column, then saves `master_df` to `combined_output.csv` inside `DATA_ROOT` (the same folder your project subfolders live in, alongside `validation_report.txt`) and confirms how many rows/columns were written. This file is the input for the quantile regression model-fitting notebook.

In [15]:
output_df = master_df.drop(
    columns=["engineering_capacity_ceng", "capacity_multiplier",
             "area_type_normalized", "project_type_category",
             "kd_year_match", "_fc_capacity", "_fc_from_name"],   # K/D source note: kept in master_df for checking, not written out
    errors="ignore",
)

comment_cols = [c for c in output_df.columns if c.strip().lower() in ("comment", "comments")]
if comment_cols:
    comment_series = output_df[comment_cols[0]]
    for extra_col in comment_cols[1:]:
        comment_series = comment_series.fillna(output_df[extra_col])
    output_df = output_df.drop(columns=comment_cols)
else:
    comment_series = pd.Series(pd.NA, index=output_df.index)
output_df["comment"] = comment_series.fillna("")

output_path = os.path.join(DATA_ROOT, "combined_output.csv")
output_df.to_csv(output_path, index=False)
print(f"Wrote {len(output_df)} rows and {len(output_df.columns)} columns to {output_path}")

Wrote 6681 rows and 49 columns to C:\Users\User\Documents\Code\forecastcards_ba_data\data\combined_output.csv
